# LoRA quickstart: data → trained adapter → served

The fastest path to a LoRA adapter this server can load and serve on the Gemma4
(`gemma4-e4b`) text tower. Two flavors of training are shown:

- **native (`nvk-train`)** — no PyTorch, no GPU, no download. Runs anywhere,
  including boxes with only CPU (e.g. `cpu-box`). The cells below execute this.
- **unsloth / HuggingFace-PEFT** — the stock ecosystem recipe. Needs a torch/GPU
  box; the produced adapter drops in here unchanged (a prefix-normalization shim
  bridges the one namespace difference). That cell is **illustrative-only** on a
  torch-less host.

> **Run every cell from the repo root** (the directory that contains `rust/` and
> `examples/`). The native cells below are byte-identical to the proven commands
> in `rust/scripts/lora-quickstart.sh` — run that script for the same flow in one
> shot. Depth: `docs/book/06.6-lora-training.md`.

## 1. Prepare data

A tiny JSONL dataset. Supported per-line schema (mix freely):
`{"ids":[u32,...]}` | `{"text":"..."}` | `{"prompt":"...","completion":"..."}`.
Swap in your own lines here — that is the only change most tasks need.

In [ ]:
%%writefile examples/lora/data.jsonl
{"text": "apple pie is good"}
{"prompt": "Bravo", "completion": " zulu nine"}
{"text": "cat dog fox run"}

## 2. Make a tiny example base model (torch-free)

A throwaway ~0.2M-param synthetic Gemma4Moe so the pipeline is demoable in ~2
minutes with no download. **For a real adapter, skip this cell** and point
`--base` in step 3 at a genuine Gemma checkpoint (a `.gguf`, or a dir with
`config.json` + `model.safetensors`).

In [ ]:
!python3 examples/lora/_make_example_base.py "$PWD/examples/lora/example-base"

## 3. Train a servable LoRA adapter (native `nvk-train`)

One command, dataset → servable adapter. Flags: `--rank`/`--alpha` (alpha defaults
to rank), `--target` a subset of `{q,k,v,o,gate,up,down}`, `--steps` AdamW steps
(`0` writes an identity adapter), `--lr`, `--seed`. Output is the standard PEFT
layout (`adapter_config.json` + `adapter_model.safetensors`).

In [ ]:
!rust/scripts/nvk-lora.sh train --base "$PWD/examples/lora/example-base" --data "$PWD/examples/lora/data.jsonl" --out "$PWD/examples/lora/out" --rank 8 --alpha 16 --target q,k,v,o,gate,up,down --steps 100 --lr 0.05 --seed 7

## 4. Check the adapter loads & routes

Validate the produced adapter loads through the real serving loader and folds its
`alpha/r` scaling. Prints `r`, `alpha`, `scaling` and the matched module count;
fails clearly on a DoRA adapter or one that matches no Gemma4 text projection.

In [ ]:
!rust/scripts/nvk-lora.sh check "$PWD/examples/lora/out"

## 5. Serve it

Point the server at the adapter directory and select it by id in the OpenAI-style
chat request `model` field:

```bash
export NV_LORA_ADAPTER_DIRS="my-lora=/path/to/examples/lora/out"
```

The server discovers any directory containing `adapter_config.json`
(`src/oapi/lora.rs`) and loads it at model-build time via `E4bLora::from_peft_dir`.
See `docs/book/06.6-lora-training.md` §1(ii) for why a stock unsloth adapter also just works.

---
## Alternative: train with unsloth (needs a torch / GPU box)

**Illustrative-only on a torch-less host** (e.g. `cpu-box` has no torch — do not run
the cell below there). Run it on your own GPU box; the resulting `my-lora/`
directory drops into steps 4–5 above unchanged. Keep `use_dora=False` — DoRA is
not supported by the serving loader and fails clearly. This is the ordinary
unsloth LoRA recipe, nothing repo-specific.

In [ ]:
# --- torch/GPU box only; illustrative on a torch-less host ---
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",   # or any Gemma-3/4 text checkpoint
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_rslora = False,          # rsLoRA is supported by the loader
    use_dora  = False,           # DoRA is NOT supported — keep this False
    target_modules = [
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
)

# ... your SFTTrainer / TRL training loop over your dataset ...

model.save_pretrained("my-lora")   # adapter_config.json + adapter_model.safetensors
# then: rust/scripts/nvk-lora.sh check my-lora   (and serve as in step 5)